In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import os


def set_fontsize(base_fontsize=15):
    fontsize = base_fontsize
    plt.rcParams.update({
        'font.size': fontsize,
        'axes.titlesize': fontsize * 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize * 0.8,
        'ytick.labelsize': fontsize * 0.8,
        'legend.fontsize': fontsize * 0.8,
        'font.family': "Arial"
    })

plt.style.use('default')

set_fontsize()

In [ ]:
@dataclass(frozen=True)
class PhaseSetting:
    label: str  
    phase_cut_date: str 


phase_settings =  [
    PhaseSetting("4 weeks", "2024-01-31"),
    PhaseSetting("8 weeks", "2024-01-03"),
    PhaseSetting("16 weeks", "2023-11-08"),
    PhaseSetting("32 weeks", "2023-07-19"),
    PhaseSetting("50 weeks", "2023-03-15"),
]


def sqlite_url_from_abs_path(db_path: Path) -> str:
    return f"sqlite:////{db_path.as_posix().lstrip('/')}"


def load_complete_trials_dataframe(
    db_path: Path,
    study_name: str,
    direction: str = "minimize",
) -> pd.DataFrame:

    storage = sqlite_url_from_abs_path(db_path)

    study = optuna.create_study(
        study_name=study_name,
        direction=direction,
        storage=storage,
        load_if_exists=True,
    )

    df = study.trials_dataframe()  # includes params/user_attrs/system_attrs columns if present
    df = df[df["state"] == "COMPLETE"].copy()

    df = df.loc[(df.user_attrs_total_nll<np.inf) & (df.user_attrs_train_nll<np.inf) & (df.user_attrs_val_nll<np.inf)]
    # Remove trials with a loss 10 times the best loss
    best_loss = df['user_attrs_val_nll'].min()
    df = df[df['user_attrs_val_nll'] <= 10 * best_loss]
    return df


def global_min_max(values: List[np.ndarray]) -> Optional[Tuple[float, float]]:
    arrs = [v[np.isfinite(v)] for v in values if v is not None and len(v) > 0]
    if not arrs:
        return None
    concat = np.concatenate(arrs)
    if concat.size == 0:
        return None
    return float(np.min(concat)), float(np.max(concat))


def visualize_hp_optimization(
    dfs: List[pd.DataFrame],
    settings: List[PhaseSetting],
    outpath: Optional[Path],
) -> None:
    rolling_bests = []
    for df in dfs:
        if df is None or df.empty:
            rolling_bests.append(np.array([]))
        else:
            rolling_bests.append(df["value"].cummin().to_numpy())

    mm = global_min_max(rolling_bests)
    use_log = False
    if mm is not None:
        vmin, vmax = mm
        use_log = (vmin > 0) and (vmax / vmin > 10)

    fig, axes = plt.subplots(2, 5, figsize=(15.1, 4.6), dpi=300, sharey=False)
    for ax, df, setting, rb in zip(axes[0,:], dfs, settings, rolling_bests):
        ax.set_title(f"{setting.label}")
        ax.set_xlabel("Trial")

        if df is None or df.empty:
            ax.text(0.5, 0.5, "No COMPLETE trials\n(or DB missing)", ha="center", va="center")
            ax.grid(True, alpha=0.3)
            continue

        x = df["number"].to_numpy()
        ax.plot(x, rb, linewidth=2)

        ax.grid(True, alpha=0.3)
        if use_log:
            ax.set_yscale("log")

    axes[0,0].set_ylabel("Best negative\nlog likelihood\n(validation)")


    trains, vals = [], []
    for df in dfs:
        if df is None or df.empty:
            trains.append(np.array([]))
            vals.append(np.array([]))
            continue

        if "user_attrs_train_nll" not in df.columns or "user_attrs_val_nll" not in df.columns:
            trains.append(np.array([]))
            vals.append(np.array([]))
            continue

        tr = pd.to_numeric(df["user_attrs_train_nll"], errors="coerce").to_numpy()
        va = pd.to_numeric(df["user_attrs_val_nll"], errors="coerce").to_numpy()
        m = np.isfinite(tr) & np.isfinite(va)
        trains.append(tr[m])
        vals.append(va[m])



    for ax, df, setting, tr, va in zip(axes[1,:], dfs, settings, trains, vals):
        ax.set_title(f"{setting.label}")
        ax.set_xlabel("Negative log\nlikelihood (training)")
        ax.scatter(tr, va, alpha=0.6)
        ax.plot([min(tr.min(), va.min()), 
                       max(tr.max(), va.max())], 
                      [min(tr.min(), va.min()), 
                       max(tr.max(), va.max())], 'r--', alpha=0.5)
        ax.grid(True, alpha=0.3)

    axes[1,0].set_ylabel("Negative\nlog likelihood\n(validation)")
    fig.tight_layout()

    if outpath is not None:
        outpath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outpath, dpi=300, bbox_inches="tight")



In [ ]:
town = "Bonn"
base_dir = Path("/home/iru-mls/marvin/ww_bonn_jax") / town
study_name = "ude_hp_search_cc_ude"
out_dir = Path("optuna_evaluation")
   
os.makedirs(out_dir, exist_ok=True)

dfs: List[pd.DataFrame] = []
missing: List[str] = []

for s in phase_settings:
    db_path = base_dir / f"optuna_study_two_phase_model_{s.phase_cut_date}.db"
    try:
        df = load_complete_trials_dataframe(
            db_path=db_path,
            study_name=study_name,
            direction="minimize",
        )
        dfs.append(df)
    except FileNotFoundError:
        dfs.append(pd.DataFrame())
        missing.append(str(db_path))

if missing:
    print("Warning: some DB files were not found:")
    for m in missing:
        print(f"  - {m}")

visualize_hp_optimization(
    dfs=dfs,
    settings=phase_settings,
    outpath=out_dir / "hp_optimization.png",
)


In [ ]:
import pandas as pd
import optuna

PHASE_CUT_DATES = [
    "2024-01-31",  # 4 weeks
    "2024-01-03",  # 8 weeks
    "2023-11-08",  # 16 weeks
    "2023-07-19",  # 32 weeks
    "2023-03-15",  # 50 weeks
]

def _sqlite_storage(town: str, phase_cut_date: str) -> str:
    return f"sqlite:////home/iru-mls/marvin/ww_bonn_jax/{town}/optuna_study_two_phase_model_{phase_cut_date}.db"

def _load_study(town: str, phase_cut_date: str, study_name: str) -> optuna.Study:
    storage = _sqlite_storage(town, phase_cut_date)
    return optuna.create_study(
        study_name=study_name,
        direction="minimize",
        storage=storage,
        load_if_exists=True,
    )

def _top_k_hp_names(study: optuna.Study, evaluator, k: int = 10) -> list[str]:
    # Returns names only, sorted by importance (descending)
    imp = optuna.importance.get_param_importances(study, evaluator=evaluator)
    names = list(imp.keys())[:k]
    if len(names) < k:
        names += [""] * (k - len(names))  # pad for rectangular dataframe
    return names

def build_hp_name_dfs(
    town: str = "Bonn",
    study_name: str = "ude_hp_search_cc_ude",
    k: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    fanova_evaluator = optuna.importance.FanovaImportanceEvaluator()
    pedanova_evaluator = optuna.importance.PedAnovaImportanceEvaluator()

    fanova_cols = {}
    pedanova_cols = {}

    for d in PHASE_CUT_DATES:
        study = _load_study(town=town, phase_cut_date=d, study_name=study_name)

        # fANOVA
        fanova_cols[d] = _top_k_hp_names(study, evaluator=fanova_evaluator, k=k)

        # PedANOVA
        pedanova_cols[d] = _top_k_hp_names(study, evaluator=pedanova_evaluator, k=k)

    idx = pd.Index(range(1, k + 1), name="rank")
    df_fanova_hp_names = pd.DataFrame(fanova_cols, index=idx)
    df_pedanova_hp_names = pd.DataFrame(pedanova_cols, index=idx)

    return df_fanova_hp_names, df_pedanova_hp_names


df_fanova_hp_names, df_pedanova_hp_names = build_hp_name_dfs(town="Bonn", study_name="ude_hp_search_cc_ude", k=10)

df_fanova_hp_names
# df_pedanova_hp_names
